In [1]:
!pip install sec-edgar-downloader PyPDF2 matplotlib nest_asyncio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [sec-edgar-downloader]


In [1]:
# Import your utility
import os
import autogen_utils 
from autogen_agentchat.agents import AssistantAgent, UserProxyAgent
from autogen_agentchat.teams import SelectorGroupChat
from autogen_agentchat.conditions import TextMentionTermination
from autogen_ext.models.openai import OpenAIChatCompletionClient


/workspace/autogen_utils.py:126: UserWarning: Using LocalCommandLineCodeExecutor may execute code on the local machine which can be unsafe. For security, it is recommended to use DockerCommandLineCodeExecutor instead. To install Docker, visit: https://docs.docker.com/get-docker/
  code_executor = LocalCommandLineCodeExecutor(work_dir="coding")


In [ ]:
os.environ["OPENAI_API_KEY"] = "YOUR_OPENAI_API_KEY"

In [3]:
from IPython.display import display, Image as IPImage

async def run_stock_mission(ticker: str, days: int, task: str, model: str = "gpt-4o"):
    model_client = OpenAIChatCompletionClient(model=model)

    # 1. THE AGENTS
    analyst = AssistantAgent(
        name="Analyst",
        model_client=model_client,
        tools=[autogen_utils.market_tool, autogen_utils.plot_tool],
        system_message="You ONLY provide raw data and charts. Do NOT interpret or give advice."    
    )

    bull = AssistantAgent(
        name="Bull_Strategist",
        model_client=model_client,
        tools=[autogen_utils.financial_tool],
        system_message="You ONLY look for positives. Speak briefly and wait for the Bear to counter."
    )

    bear = AssistantAgent(
        name="Bear_Strategist",
        model_client=model_client,
        tools=[autogen_utils.financial_tool],
        system_message="You ONLY look for risks. Speak briefly and challenge the Bull."    
    )

    executive = UserProxyAgent(name="Executive")

    # 2. THE SELECTOR TEAM
    team = SelectorGroupChat(
        [analyst, bull, bear, executive],
        model_client=model_client, 
        termination_condition=TextMentionTermination("TERMINATE")
    )
    librarian = AssistantAgent(
        name="Librarian",
        model_client=model_client,
        tools=[autogen_utils.fetch_tool, autogen_utils.pdf_converter_tool], # It can now fetch AND convert
        system_message="""
        1. First, fetch the latest 10-K for the requested ticker.
        2. Once downloaded, convert that file into a text file in 'report_texts/'.
        3. Hand off to the Analyst once the text file is ready.
        """
    )
    print(f"\n🚀 DYNAMIC DEBATE: {ticker.upper()}")
    print("=" * 45)

    # 3. MESSAGE STREAM
    history = [] 
    async for message in team.run_stream(task=task): # Using the passed 'task'
        history.append(message) 
        
        msg_type = str(type(message)).lower()
        if "toolcall" in msg_type or "tool_response" in msg_type:
            continue
            
        source = message.source.upper()
        
        if hasattr(message, 'content'):
            content = message.content
            
            # --- IMAGE RENDERING ---
            if "autogen_core" in str(content):
                if source == "ANALYST": 
                    print(f"\n[{source}]: 📊 Rendering market trend chart...")
                    display(IPImage(filename=f"{ticker.upper()}_chart.png"))
                continue

            # --- TEXT RENDERING ---
            text = str(content).strip()
            if text and not text.startswith("ToolResponse"):
                print(f"\n[{source}]: {text}")

        if "TERMINATE" in str(message.content).upper():
            break
            
    return history

In [5]:
# User Inputs
ticker_in = input("Ticker: ").upper().strip()
days_in = int(input("Days: "))

print("\n--- Mission Options ---")
print("1. Standard Debate (Press Enter)")
print("2. Focus only on the Bear case")
print("3. Compare this stock to industry peers")
user_choice = input("Select 1, 2, or 3: ").strip()

# Map the numbers to actual instructions
tasks = {
    "1": f"Analyst: Provide data for {ticker_in} ({days_in} days). Bull and Bear: Debate the value.",
    "2": f"Analyst: Provide data. Bear_Strategist: Give a deep dive on every risk you find.",
    "3": f"Analyst: Provide data. Bull and Bear: Debate how {ticker_in} stacks up against its main competitors."
}

# If they just hit enter or type something else
final_task = tasks.get(user_choice, f"Debate {ticker_in} based on {days_in} days of data.")

# Execute
history = await run_stock_mission(ticker=ticker_in, days=days_in, task=final_task)

Ticker:  NVDA
Days:  100



--- Mission Options ---
1. Standard Debate (Press Enter)
2. Focus only on the Bear case
3. Compare this stock to industry peers


Select 1, 2, or 3:  1



🚀 DYNAMIC DEBATE: NVDA

[USER]: Analyst: Provide data for NVDA (100 days). Bull and Bear: Debate the value.


Enter your response:  Fetch the details of the company using the library agent



[EXECUTIVE]: Fetch the details of the company using the library agent

[ANALYST]: I don't have access to external databases for company details or library resources directly here. If you have specific data points or information you're looking for, please let me know, and I can help gather or analyze available data.


Enter your response:  terminate



[EXECUTIVE]: terminate


In [14]:
ticker_in = input("Enter Ticker (eg. TSLA): ").upper().strip()
days_in = int(input("Enter number of Days to fetch the financial data of the stock: "))

# 2. Run directly with await
await run_stock_mission(ticker=ticker_in, days=days_in)

Enter Ticker (eg. TSLA):  TSLA
Enter number of Days to fetch the financial data of the stock:  90


TypeError: run_stock_mission() missing 1 required positional argument: 'task'

In [19]:
from autogen_agentchat.agents import AssistantAgent, CodeExecutorAgent
from autogen_agentchat.teams import Swarm # Swarm is better for Handoffs
from autogen_agentchat.conditions import HandoffTermination

async def run_advanced_mission(ticker: str, pdf_path: str):
    model_client = OpenAIChatCompletionClient(model="gpt-4o")

    # 1. RAG AGENT (The Librarian)
    librarian = AssistantAgent(
        name="Librarian",
        model_client=model_client,
        tools=[pdf_tool],
        system_message="Read the 10-K report. Summarize 3 risks. When done, hand off to 'Quant'."
    )

    # 2. CODE AGENT (The Quant)
    # This agent writes code. The CodeExecutorAgent runs it.
    quant_coder = AssistantAgent(
        name="Quant_Coder",
        model_client=model_client,
        system_message="Write Python code to run a Monte Carlo simulation for 1000 paths for the stock. Output the final mean. When done, hand off to 'Compliance'."
    )
    
    executor_agent = CodeExecutorAgent(
        name="Executor",
        code_executor=code_executor
    )

    # 3. COMPLIANCE AGENT (State 3)
    compliance = AssistantAgent(
        name="Compliance",
        model_client=model_client,
        system_message="Review the Librarian's risks and the Quant's math. Give a final 'CLEARED' or 'REJECTED' verdict."
    )

    # Define the "Swarm" with Handoffs
    # Librarian -> Quant -> Compliance
    team = Swarm(
        participants=[librarian, quant_coder, executor_agent, compliance],
        termination_condition=TextMentionTermination("CLEARED")
    )

    print(f"🚀 STARTING ADVANCED WORKFLOW FOR {ticker}")
    await team.run(task=f"Start with Librarian reading {pdf_path}. Then run Quant simulation.")

In [22]:
# --- Step 1: Initialize the Workspaces ---
import os
os.makedirs("coding", exist_ok=True)

# --- Step 2: Define the Execution Call ---
async def main_demo():
    # Provide the ticker and the path to your uploaded 10-K PDF
    ticker = "NVDA"
    pdf_path = "NVDA_chart.png" 
    
    if not os.path.exists(pdf_path):
        print(f"⚠️ Warning: {pdf_path} not found. RAG tool will fail unless file is present.")
    
    # --- Step 3: Trigger the Advanced Mission ---
    # This calls the Swarm/Handoff logic we defined
    history = await run_advanced_mission(
        ticker=ticker, 
        pdf_path=pdf_path
    )
    
    print("\n✅ Advanced Workflow Complete.")

# --- Step 4: Run it ---
import asyncio
if __name__ == "__main__":
    # If running in a script:
    # asyncio.run(main_demo())
    
    # If running in a Jupyter Notebook:
    await main_demo()

NameError: name 'pdf_tool' is not defined